In [ ]:
import re


# 평균 문장 길이 계산에서 제거할 단독 채움 표현
FILLER_WORDS_FOR_LENGTH = {
    "어어", "어어어",
    "음", "음음", "으음", "으으음",
    "에", "에에",
    "흠", "흐음"
}


# 평균 문장 길이 계산에서 제거할 구 단위 채움 표현
FILLER_PHRASES_FOR_LENGTH = {
    "그 뭐냐",
    "그 뭐지",
    "그 뭐더라",
    "그 뭐였더라",
    "어 뭐냐",
    "어 뭐지",
    "어 뭐더라",
    "음 뭐냐",
    "음 뭐지",
    "음 뭐더라",
    "뭐라고 해야 되나",
    "뭐라고 해야 하나",
    "뭐라 해야 되나",
    "뭐라 해야 하나",
    "뭐라고 그러지",
    "뭐라 그러지",
    "그 뭐시냐",
    "그 머시냐",
    "그 뭐시더라",
    "그 머시더라",
    "뭐시더라",
    "머시더라",
}


def remove_filler_phrases_for_length(text):
    """
    평균 문장 길이 계산용으로 구 단위 채움 표현을 제거한다.
    예: '그 뭐냐', '뭐라고 해야 하나'
    """

    if text is None:
        return ""

    text = str(text)

    # 긴 표현부터 제거해야 짧은 표현과 충돌이 덜 난다.
    phrases = sorted(FILLER_PHRASES_FOR_LENGTH, key=len, reverse=True)

    for phrase in phrases:
        text = text.replace(phrase, " ")

    text = re.sub(r"\s+", " ", text).strip()

    return text


def remove_filler_words_for_length(text):
    """
    평균 문장 길이 계산용으로 단독 채움 표현을 제거한다.
    '어', '음' 등이 독립된 토큰일 때만 제거한다.
    """

    if text is None:
        return ""

    text = str(text).strip()

    if text == "":
        return ""

    tokens = text.split()
    kept_tokens = []

    for token in tokens:
        # 양끝 문장부호만 제거해서 비교한다.
        # 예: '어,' → '어'
        clean_token = re.sub(r"^[.,!?。！？…~\"'“”‘’()\[\]{}<>]+", "", token)
        clean_token = re.sub(r"[.,!?。！？…~\"'“”‘’()\[\]{}<>]+$", "", clean_token)

        if clean_token in FILLER_WORDS_FOR_LENGTH:
            continue

        kept_tokens.append(token)

    return " ".join(kept_tokens)


def remove_fillers_for_length(text):
    """
    평균 문장 길이 계산용으로만 채움 표현을 제거한다.
    원본 STT나 preprocessed_text 자체를 바꾸는 용도가 아니다.
    """

    text = remove_filler_phrases_for_length(text)
    text = remove_filler_words_for_length(text)

    text = re.sub(r"\s+", " ", text).strip()

    return text


def count_meaningful_chars(text):
    """
    답변 전체 의미 글자 수를 계산한다.
    공백과 문장부호를 제외한다.
    """

    if text is None:
        return 0

    text = str(text)

    # 공백 제거
    text = re.sub(r"\s+", "", text)

    # 문장부호 및 괄호류 제거
    text = re.sub(r"[.,!?。！？…~\"'“”‘’()\[\]{}<>]", "", text)

    return len(text)


def calculate_answer_length(text):
    """
    답변 전체 의미 글자 수를 계산한다.
    기존 avg_sentence_length 컬럼에 저장하더라도,
    실제 의미는 '답변 전체 의미 글자 수'로 정의한다.
    """

    if text is None:
        return 0.0

    text = str(text).strip()

    if text == "":
        return 0.0

    cleaned_text = remove_fillers_for_length(text)
    char_count = count_meaningful_chars(cleaned_text)

    return float(char_count)

In [ ]:
def score_answer_length(answer_length, question_type_name):
    """
    답변 전체 의미 글자 수를 0~100점으로 변환한다.
    answer_length: 채움 표현, 공백, 문장부호를 제외한 글자 수
    """

    if answer_length is None or answer_length <= 0:
        return 0

    # 오늘 날짜 말하기는 짧게 답해도 정상.
    # 길이는 최종 점수에 반영하지 않거나, 발화 존재 여부만 본다.
    if question_type_name == "오늘 날짜 말하기":
        return 100

    # 공통점 말하기는 짧아도 정답일 수 있음.
    # 예: 둘 다 과일이에요 → 7글자
    if question_type_name == "규칙 기반 언어추론":
        if answer_length >= 6:
            return 100
        elif answer_length >= 3:
            return 80
        elif answer_length >= 1:
            return 60
        else:
            return 0

    # 상황 질문은 행동이 드러나면 짧아도 가능.
    # 예: 119를 불러요 → 6~7글자
    if question_type_name == "상황 질문 답하기":
        if answer_length >= 10:
            return 100
        elif answer_length >= 6:
            return 80
        elif answer_length >= 3:
            return 60
        elif answer_length >= 1:
            return 40
        else:
            return 0

    # 그림 설명은 여러 요소를 묘사해야 하므로 조금 더 긴 발화가 자연스러움.
    if question_type_name == "그림 설명하기":
        if answer_length >= 25:
            return 100
        elif answer_length >= 15:
            return 80
        elif answer_length >= 6:
            return 60
        elif answer_length >= 1:
            return 40
        else:
            return 0

    # 추억 말하기는 구체적 회상이 필요하므로 가장 길이 기준을 높게 둔다.
    if question_type_name == "추억 말하기":
        if answer_length >= 30:
            return 100
        elif answer_length >= 18:
            return 80
        elif answer_length >= 8:
            return 60
        elif answer_length >= 1:
            return 40
        else:
            return 0

    return 0

In [ ]:
def analyze_answer_length(text, question_type_name):
    """
    답변 전체 의미 글자 수와 길이 점수를 함께 계산한다.
    """

    answer_length = calculate_answer_length(text)
    answer_length_score = score_answer_length(answer_length, question_type_name)

    return {
        "avg_sentence_length": answer_length,  # DB 컬럼명이 avg_sentence_length라면 여기에 저장
        "sentence_length_score": answer_length_score
    }

In [ ]:
samples = [
    {
        "text": "어 음 그 뭐냐 수도꼭지가 안 나오면 관리실에 연락해야제.",
        "question_type": "상황 질문 답하기"
    },
    {
        "text": "음 뭐라고 해야 하나, 일단 옆 사람한테 좀 도와달라 해야겠네.",
        "question_type": "상황 질문 답하기"
    },
    {
        "text": "어어 그 뭐더라, 병원에 전화하고 보호자한테 알려야 쓰겄네.",
        "question_type": "상황 질문 답하기"
    },
    {
        "text": "그 뭐시냐 시장에 사람이 많고 아지매가 생선을 손질하고 있구먼.",
        "question_type": "그림 설명하기"
    },
    {
        "text": "음음 장에서 손님들이 물건 고르고, 상인은 저울에 뭘 달고 있는 것 같어.",
        "question_type": "그림 설명하기"
    },
    {
        "text": "어 뭐지 부엌에서 국이 넘치고 애가 그릇을 떨어뜨린 것 같네잉.",
        "question_type": "그림 설명하기"
    },
    {
        "text": "어릴 적에는 말이여, 할매가 해주던 고구마죽을 참 맛나게 먹었제.",
        "question_type": "추억 말하기"
    },
    {
        "text": "음 그 뭐냐, 장날마다 아버지 따라가서 꽈배기 하나 얻어먹던 기억이 나네.",
        "question_type": "추억 말하기"
    },
    {
        "text": "그 머시더라, 학교 가는 길에 개울 건너다가 신발 젖었던 일이 있었어.",
        "question_type": "추억 말하기"
    },
    {
        "text": "어어 사과하고 바나나는 둘 다 먹는 과일 아니겄어.",
        "question_type": "규칙 기반 언어추론"
    },
    {
        "text": "음음 연필하고 볼펜은 글씨 쓰는 데 쓰는 거제.",
        "question_type": "규칙 기반 언어추론"
    },
    {
        "text": "으음 오늘은 유월 사일 목요일인 것 같은디.",
        "question_type": "오늘 날짜 말하기"
    },
]

for sample in samples:
    cleaned = remove_fillers_for_length(sample["text"])
    result = analyze_answer_length(sample["text"], sample["question_type"])

    print("문항 유형:", sample["question_type"])
    print("원문:", sample["text"])
    print("길이 계산용:", cleaned)
    print("의미 글자 수:", result["avg_sentence_length"])
    print("길이 점수:", result["sentence_length_score"])
    print("-" * 60)

문항 유형: 상황 질문 답하기
원문: 어 음 그 뭐냐 수도꼭지가 안 나오면 관리실에 연락해야제.
길이 계산용: 어 수도꼭지가 안 나오면 관리실에 연락해야제.
의미 글자 수: 19.0
길이 점수: 100
------------------------------------------------------------
문항 유형: 상황 질문 답하기
원문: 음 뭐라고 해야 하나, 일단 옆 사람한테 좀 도와달라 해야겠네.
길이 계산용: , 일단 옆 사람한테 좀 도와달라 해야겠네.
의미 글자 수: 16.0
길이 점수: 100
------------------------------------------------------------
문항 유형: 상황 질문 답하기
원문: 어어 그 뭐더라, 병원에 전화하고 보호자한테 알려야 쓰겄네.
길이 계산용: , 병원에 전화하고 보호자한테 알려야 쓰겄네.
의미 글자 수: 18.0
길이 점수: 100
------------------------------------------------------------
문항 유형: 그림 설명하기
원문: 그 뭐시냐 시장에 사람이 많고 아지매가 생선을 손질하고 있구먼.
길이 계산용: 시장에 사람이 많고 아지매가 생선을 손질하고 있구먼.
의미 글자 수: 22.0
길이 점수: 80
------------------------------------------------------------
문항 유형: 그림 설명하기
원문: 음음 장에서 손님들이 물건 고르고, 상인은 저울에 뭘 달고 있는 것 같어.
길이 계산용: 장에서 손님들이 물건 고르고, 상인은 저울에 뭘 달고 있는 것 같어.
의미 글자 수: 26.0
길이 점수: 100
------------------------------------------------------------
문항 유형: 그림 설명하기
원문: 어 뭐지 부엌에서 국이 넘치고 애가 그릇을 떨어뜨린 것 같네잉.
길이 계산용: 부엌에서 국이 넘치고 애가 그릇을 떨어뜨린 것 같네잉.
의미 글자 수: 